# 04 — LSTM Deep Learning Model



In [1]:
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent))

from config.settings import (
    DENSE_UNITS,
    DROPOUT_RATE,
    EARLY_STOPPING_PATIENCE,
    EMBEDDING_DIM,
    EXPERIMENTS_LOG_FILE,
    LSTM_BATCH_SIZE,
    LSTM_MAX_EPOCHS,
    LSTM_MODEL_DIR,
    LSTM_UNITS,
    MAX_SEQUENCE_LENGTH,
    OOV_TOKEN,
    REPORT_FIGURES_DIR,
    REPORT_TABLES_DIR,
    SPATIAL_DROPOUT_RATE,
    TOP_N_FEATURES,
    VOCAB_SIZE,
)
from training.lstm import (
    build_lstm_model,
    build_tokenizer,
    load_lstm_dataset,
    save_lstm_artifacts,
    set_random_seeds,
    texts_to_padded_sequences,
    train_lstm_model,
)
from training.split import stratified_three_way_split
from evaluation.metrics import (
    compute_classification_metrics,
    plot_confusion_matrix,
    plot_roc_curve,
    save_classification_report,
)
from evaluation.error_analysis import get_false_negatives, get_false_positives
from evaluation.experiment_log import log_experiment
from evaluation.training_history import plot_training_history

pd.set_option("display.max_colwidth", 100)
REPORT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

set_random_seeds()


## Before writing any code: how does an LSTM text classifier actually work?

The baseline treated an article as an unordered "bag of words" — TF-IDF has no notion of which
word came before which. An **LSTM (Long Short-Term Memory)** network is different: it reads a
sequence of words **one at a time, in order**, carrying forward a memory of everything it has
read so far. This is exactly the capability the baseline's Error Analysis (`docs/baseline_model_report.md`)
identified as missing — the ability to use word order and negation (`"did not confirm"` vs.
`"did confirm"`).

### Why LSTM instead of a plain RNN?

A plain Recurrent Neural Network (RNN) also reads sequences in order, but it suffers badly from
the **vanishing gradient problem**: during training, the error signal used to update earlier
words' influence shrinks exponentially as it's propagated backward through many time steps,
so a plain RNN effectively "forgets" anything more than a few words back. An LSTM fixes this
with a more elaborate internal cell containing three **gates** — forget, input, and output —
that explicitly control what to keep, what to add, and what to expose from a separate "cell
state" that can carry information across many more time steps without vanishing. This is
covered word-by-word in `docs/viva_notes.md`'s Phase 5 section; the short version used here is:
**LSTM was chosen specifically to avoid the vanishing gradient problem that makes plain RNNs
impractical for anything but very short sequences**, and this dataset's articles run to
hundreds of words.

### The full pipeline built in this notebook

```
lstm_text (already cleaned in Stage 3: lowercase, HTML/URL/punctuation removed,
           stop words + lemmatization deliberately NOT applied - docs/preprocessing_plan.md)
      ↓
Tokenizer (word -> integer index, built ONLY from the training set)
      ↓
Padding/truncating to a fixed length (MAX_SEQUENCE_LENGTH)
      ↓
Embedding layer (integer index -> dense vector, learned during training)
      ↓
SpatialDropout1D (regularization on the embedding output)
      ↓
LSTM layer (reads the sequence, produces one summary vector)
      ↓
Dropout (regularization)
      ↓
Dense(ReLU) -> Dense(Sigmoid) (final classification head)
```

Each stage is explained in its own section below, immediately before the code that implements
it — no code is written without an explanation first.


## Step 1 — Load data and reuse the identical split

`load_lstm_dataset()` (in `training/lstm.py`) keeps only `title`, `text`, `label`, and
`lstm_text` — structurally dropping `baseline_text` at the source, the same safeguard
`training/baseline.py` uses in reverse.

`stratified_three_way_split()` is the **exact same function, same `RANDOM_SEED`, same 70/15/15
ratio** used for the baseline in Phase 4. Because the row order and `label` column in
`03_preprocessed.csv` are unchanged, this reproduces the identical train/validation/test rows
the baseline was evaluated on — required for the comparison in Phase 6 to be fair (and already
verified working exactly this way in `docs/reuters_ablation_study.md`).


In [2]:
df = load_lstm_dataset()
print(f"Rows: {len(df):,}   Columns: {list(df.columns)}")

train_df, val_df, test_df = stratified_three_way_split(df)
print(f"Train: {len(train_df):,}   Validation: {len(val_df):,}   Test: {len(test_df):,}")

y_train = train_df["label"].values
y_val = val_df["label"].values
y_test = test_df["label"].values


Rows: 38,638   Columns: ['title', 'text', 'label', 'lstm_text']
Train: 27,046   Validation: 5,796   Test: 5,796


## Step 2 — Tokenization: what is it, and why do we need it?

**What is a vocabulary, in this context?** The complete set of distinct words the model is
allowed to recognize — every word outside this set gets mapped to a single "unknown word"
placeholder. **What is tokenization?** Splitting text into individual word tokens and assigning
each a unique integer ID — `"trump said no"` might become `{"trump": 42, "said": 7, "no": 15}`
then the sequence `[42, 7, 15]`.

This is conceptually similar to TF-IDF's vocabulary building in Phase 4, but the output is
different: TF-IDF produces one score per vocabulary word (a bag-of-words vector); a Tokenizer
produces one integer ID **per word position**, preserving order — exactly the extra information
an LSTM needs and TF-IDF discards.

**Why cap the vocabulary at `VOCAB_SIZE` (20,000)?** Without a cap, every rare word, typo, or
one-off token would get its own row in the Embedding layer's weight matrix, most of which would
never see enough training examples to learn a meaningful vector. Capping at the 20,000 most
frequent words (via Keras `Tokenizer(num_words=...)`) and mapping everything else to
`OOV_TOKEN = "<OOV>"` keeps the embedding table a manageable size and avoids wasting model
capacity on words the model will rarely or never see again.

**Why fit the tokenizer on the training set only?** The exact same reproducibility rule as the
baseline's TF-IDF vectorizer (`docs/preprocessing_plan.md`): fitting on the full dataset before
splitting would let validation/test vocabulary quietly influence the model's word index — a
second, subtler form of train/test leakage beyond the row-level duplicate leakage already
handled in `docs/duplicate_analysis.md`.


In [3]:
tokenizer = build_tokenizer(train_df["lstm_text"])

actual_vocab_size = len(tokenizer.word_index)
print(f"Distinct words seen in training data: {actual_vocab_size:,}")
print(f"Vocabulary cap (VOCAB_SIZE): {VOCAB_SIZE:,}")
print(f"OOV token: '{OOV_TOKEN}'")
print()
print("Most frequent 10 words:", list(tokenizer.word_index.items())[:10])


Distinct words seen in training data: 96,249
Vocabulary cap (VOCAB_SIZE): 20,000
OOV token: '<OOV>'

Most frequent 10 words: [('<OOV>', 1), ('the', 2), ('to', 3), ('of', 4), ('a', 5), ('and', 6), ('in', 7), ('s', 8), ('that', 9), ('on', 10)]


**Observation:** the training data contains more distinct words than the 20,000 cap
(the Tokenizer already learned an index for every word it saw — the `num_words` cap is applied
later, at `texts_to_sequences()` time, dropping any word whose index is beyond 20,000).


## Step 3 — Sequence padding: what is it, and why do we need it?

Articles have different lengths, but a neural network layer expects every input in a batch to
have the **same fixed shape**. **Padding** solves this by forcing every sequence to exactly
`MAX_SEQUENCE_LENGTH` (300) integers: shorter sequences get zeros appended at the end
("post"-padding); longer sequences get cut off at 300 ("post"-truncating). Padding/truncating
at the *end* rather than the *start* keeps each article's opening sentences — typically where
news writing front-loads the most important information — intact in both cases.

**Why 300 specifically?** Chosen in Phase 2 EDA (`01_dataset_analysis.ipynb`) based on the
*median* article length (~360 words before Stage 3 cleaning, slightly less after), not the
maximum (~8,100 words) — using the maximum would waste enormous amounts of computation padding
nearly every article to a length almost none of them need.


In [4]:
X_train = texts_to_padded_sequences(tokenizer, train_df["lstm_text"])
X_val = texts_to_padded_sequences(tokenizer, val_df["lstm_text"])
X_test = texts_to_padded_sequences(tokenizer, test_df["lstm_text"])

print(f"X_train shape: {X_train.shape}  (rows x MAX_SEQUENCE_LENGTH)")
print("Example padded sequence (first 20 values):", X_train[0][:20])
print("Example padded sequence (last 20 values):", X_train[0][-20:])


X_train shape: (27046, 300)  (rows x MAX_SEQUENCE_LENGTH)
Example padded sequence (first 20 values): [  793  9101     8   704   248 12012 18346    13     8  7852   139   202
   177     3     1   743   793  9101    22    45]
Example padded sequence (last 20 values): [  44   77 3980   16    1   13  212  285   99   15 2176   52 1248  104
 4213   20 3689   12  566   47]


**Observation:** unlike the baseline's TF-IDF matrix (one row per article, one column
per *vocabulary word*, mostly zero), this matrix has one row per article and exactly 300
columns, one per *word position* — the trailing zeros are padding for an article shorter than
300 words.


## Step 4 — Model architecture

```
Embedding(vocab_size=20,000, embedding_dim=100)
      ↓
SpatialDropout1D(0.2)
      ↓
LSTM(64 units)
      ↓
Dropout(0.3)
      ↓
Dense(32, activation="relu")
      ↓
Dense(1, activation="sigmoid")
```

**Embedding layer — why do we need it, and what does it do?** The Tokenizer's integer IDs are
arbitrary — word 42 is not "more" or "less" than word 7 in any meaningful sense, so feeding raw
integers directly into a network would imply a false numeric relationship. The Embedding layer
instead **learns** a dense vector (length `EMBEDDING_DIM = 100`) for every vocabulary word
during training, positioning semantically similar words closer together in that 100-dimensional
space. This is the direct Deep Learning analogue of TF-IDF's scoring — both turn words into
numbers — but an embedding is *learned* and *dense* (100 real numbers per word), while TF-IDF is
*computed* and *sparse* (one 0-or-nonzero score per vocabulary word, no notion of similarity
between words).

**SpatialDropout1D — why include it, and why not plain Dropout?** Regular `Dropout` randomly
zeroes out individual numbers; `SpatialDropout1D` instead randomly zeroes out **entire embedding
channels** (whole dimensions of the embedding vector) consistently across a whole sequence. For
word embeddings specifically, this is the more appropriate form of regularization: it prevents
the LSTM from becoming overly reliant on any single embedding dimension, rather than corrupting
scattered individual numbers inside otherwise-intact word vectors. Included here (marked
optional in the project brief) because embeddings over a 20,000-word vocabulary with a
moderately-sized training set (~27,000 articles) are a realistic overfitting risk worth guarding
against directly at the embedding output.

**LSTM layer — what does it actually do?** Reads the 300-step sequence of embedding vectors one
step at a time. At each step, three gates decide what happens to its internal "cell state" (a
running memory):
- **Forget gate:** decides what to discard from the memory so far.
- **Input gate:** decides what new information from the current word to add.
- **Output gate:** decides what part of the (updated) memory to expose as this step's output.

After all 300 steps, the LSTM layer (used here without `return_sequences=True`) returns just the
**final** hidden state — one 64-number vector summarizing the whole article — which is what
feeds into the classification head below.

**Dropout — why again, after the LSTM?** A second, ordinary Dropout on the LSTM's output vector,
guarding against overfitting in the dense classification head specifically, independent of the
embedding-level regularization above.

**Dense(ReLU) then Dense(Sigmoid) — why two dense layers, and why these activations?** The first
Dense layer (32 units, ReLU activation) lets the model combine the LSTM's summary vector
non-linearly before the final decision — ReLU (`max(0, x)`) is the standard default activation
for hidden layers: simple, fast, and avoids the vanishing-gradient issues associated with older
activations like sigmoid/tanh in deep stacks. The final `Dense(1, activation="sigmoid")` layer
is what actually makes this a *binary* classifier: sigmoid squashes its output into (0, 1),
directly interpretable as "probability the article is Real" — the same interpretation the
baseline's Logistic Regression output has, which is exactly what makes the two models' outputs
comparable in Phase 6.

**Why not BiLSTM / Attention / CNN-LSTM / Transformers?** All are legitimate techniques that
could plausibly improve on a plain LSTM, but each adds architectural complexity, more
hyperparameters, and harder-to-explain internals — working against this phase's stated
priorities (correctness, reproducibility, educational value, clean architecture) and against
the project specification's "avoid advanced architectures unless there is a clear academic benefit" rule. A
plain, unidirectional LSTM is simple enough to fully explain in a viva end to end, which a
Transformer's self-attention mechanism is a much harder claim to make for a BCA-level defense.


In [5]:
model = build_lstm_model()
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 300, 100)       │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 300, 100)       │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        42,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,044,353 (7.80 MB)

 Trainable params: 2,044,353 (7.80 MB)

 Non-trainable params: 0 (0.00 B)

**Why `binary_crossentropy` as the loss function, and `adam` as the optimizer?**
Binary crossentropy directly measures how far off a predicted probability is from the true 0/1
label, penalizing confident wrong predictions much more heavily than uncertain ones — the
standard, correct loss for a single-output sigmoid binary classifier (using anything else, e.g.
mean squared error, would give the model weaker gradient signal for exactly the confidently-wrong
predictions it most needs to correct). `adam` is used as the optimizer — an established,
low-maintenance default that adapts its own learning rate per parameter, avoiding the need to
hand-tune a learning rate schedule for a first LSTM implementation.


## Step 5 — Training with callbacks

**EarlyStopping:** monitors validation loss; if it doesn't improve for `EARLY_STOPPING_PATIENCE`
(3) consecutive epochs, training stops and the model's weights are rolled back
(`restore_best_weights=True`) to whichever epoch actually had the lowest validation loss — not
necessarily the last epoch trained. This directly guards against overfitting: without it, the
model could keep improving on the training set while quietly getting worse on unseen data, and
naively using the final epoch's weights would ship the overfit version.

**ModelCheckpoint:** independently saves the best-validation-loss model to disk during training,
as a safety net (e.g. if training were interrupted).

Up to `LSTM_MAX_EPOCHS` (10) epochs are allowed; in practice, EarlyStopping is expected to stop
training earlier once validation loss stops improving.


In [6]:
start_time = time.time()
history = train_lstm_model(model, X_train, y_train, X_val, y_val, LSTM_MODEL_DIR / "model.keras")
training_time_seconds = time.time() - start_time

epochs_completed = len(history.history["loss"])
best_val_loss = min(history.history["val_loss"])
best_val_accuracy = max(history.history["val_accuracy"])

print(f"Training time: {training_time_seconds:.1f} seconds")
print(f"Epochs completed: {epochs_completed} (max allowed: {LSTM_MAX_EPOCHS})")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Best validation accuracy: {best_val_accuracy:.4f}")


Epoch 1/10


423/423 - 72s - 171ms/step - accuracy: 0.7876 - loss: 0.4835 - val_accuracy: 0.8087 - val_loss: 0.4947


Epoch 2/10


423/423 - 66s - 155ms/step - accuracy: 0.8468 - loss: 0.3887 - val_accuracy: 0.8444 - val_loss: 0.3740


Epoch 3/10


...

Epoch 9/10


423/423 - 57s - 136ms/step - accuracy: 0.9501 - loss: 0.1516 - val_accuracy: 0.9703 - val_loss: 0.1104


Epoch 10/10


423/423 - 57s - 135ms/step - accuracy: 0.8566 - loss: 0.3067 - val_accuracy: 0.9619 - val_loss: 0.1167


Training time: 599.1 seconds
Epochs completed: 10 (max allowed: 10)
Best validation loss: 0.1100
Best validation accuracy: 0.9703


In [7]:
plot_training_history(history, REPORT_FIGURES_DIR / "lstm_training_history.png")
plt.imread(REPORT_FIGURES_DIR / "lstm_training_history.png").shape  # confirm it saved


(675, 1650, 4)

**Reading the training-history plot:** if the training curve keeps improving while the
validation curve flattens or worsens, that's overfitting in progress — exactly what
EarlyStopping is designed to catch and roll back from. A validation curve that tracks the
training curve closely (as expected here, given the regularization via SpatialDropout1D and
Dropout) indicates the model is generalizing reasonably, not just memorizing.


## Step 6 — Evaluation

Every function below is reused **unchanged** from `evaluation/metrics.py` — the same code that
scored the baseline in Phase 4. This is only possible because that module was written to be
model-agnostic from the start (it takes `y_true`/`y_pred`/`y_proba`, not a specific model type).


In [8]:
y_proba_test = model.predict(X_test).flatten()
y_pred_test = (y_proba_test >= 0.5).astype(int)

metrics = compute_classification_metrics(y_test, y_pred_test)
for name, value in metrics.items():
    print(f"{name:>10}: {value:.4f}")



  1/182 ━━━━━━━━━━━━━━━━━━━━ 29s 165ms/step


  3/182 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step  


  5/182 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step


  7/182 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step


  9/182 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step

...


179/182 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


181/182 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step


182/182 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step


182/182 ━━━━━━━━━━━━━━━━━━━━ 5s 29ms/step


  accuracy: 0.9720
 precision: 0.9757
    recall: 0.9733
  f1_score: 0.9745


In [9]:
report_df = save_classification_report(
    y_test, y_pred_test, REPORT_TABLES_DIR / "lstm_classification_report.csv"
)
report_df


,precision,recall,f1-score,support
Fake,0.967619,0.970577,0.969096,2617.00000
Real,0.975717,0.973262,0.974488,3179.00000
accuracy,0.972050,0.972050,0.972050,0.97205
macro avg,0.971668,0.971920,0.971792,5796.00000
weighted avg,0.972061,0.972050,0.972053,5796.00000


In [10]:
plot_confusion_matrix(
    y_test, y_pred_test,
    REPORT_FIGURES_DIR / "lstm_confusion_matrix.png",
    "LSTM — Confusion Matrix",
)
auc_score = plot_roc_curve(
    y_test, y_proba_test,
    REPORT_FIGURES_DIR / "lstm_roc_curve.png",
    "LSTM — ROC Curve",
)
print(f"AUC: {auc_score:.4f}")


AUC: 0.9917


## Step 7 — Error Analysis

Reused unchanged from `evaluation/error_analysis.py` (the same functions used for the baseline
in Phase 4).


In [11]:
false_positives = get_false_positives(test_df, y_test, y_pred_test, n=5)
false_negatives = get_false_negatives(test_df, y_test, y_pred_test, n=5)

print(f"False positives (Fake predicted as Real): {int(((y_test==0) & (y_pred_test==1)).sum())}")
print(f"False negatives (Real predicted as Fake): {int(((y_test==1) & (y_pred_test==0)).sum())}")


False positives (Fake predicted as Real): 77
False negatives (Real predicted as Fake): 85


In [12]:
false_positives[["title"]]


,title
198,Trump Condemned By Jewish Leaders In Poland After Snubbing Warsaw Ghetto Memorial
220,MITCH MCCONNELL: The Senate Will Not Take Up Nomination of Merrick Garland
277,ANOTHER WIN FOR TRUMP! Release Of Americans Jailed In Egypt Secured By Trump Administration [Video]
305,Conservatives Freak Out After Seeing Obama In Front Of Che Guevara Mural (TWEETS)
417,Benghazi Survivor On Hillary Clinton: “I Don’t Think She Has A Soul”


In [13]:
false_negatives[["title"]]


,title
89,Merkel scolds ally to shield coalition talks from weedkiller row
229,Obama warns Democrats against overconfidence about Clinton victory
263,"Orlando shooter traveled to Saudi Arabia in 2011, 2012: MSNBC"
293,Danish fishermen could be hit hard by Brexit: research report
418,BOJ governor Kuroda warns against policies unwinding free trade


**Comparing with the baseline's errors:** the baseline's Error Analysis
(`notebooks/03_baseline_model.ipynb`) found false positives concentrated in Fake articles
written in an atypically neutral register, and false negatives in Real articles on politically
charged or opinion-adjacent subject matter — both explained as the baseline keying on writing
style/topic rather than genuine fact-checking. Read the LSTM's misclassified examples above
against that same lens: if similar titles/topics reappear here, that's evidence the two models
share a common ceiling (the dataset's inherent style/topic confound, see
`docs/reuters_ablation_study.md`); if the LSTM's errors look qualitatively different, that
would suggest it is picking up on different (possibly more sequence-dependent) cues than the
baseline. A full side-by-side comparison is deferred to Phase 6, once both models' complete
error sets can be directly cross-referenced.


## Experiment Tracking

Appended to `evaluation/experiments.csv` via the same `log_experiment()` function used for the
baseline and the Reuters ablation experiment — appends only, never overwrites earlier rows.


In [14]:
experiment_record = {
    "timestamp": pd.Timestamp.utcnow().isoformat(),
    "model": "LSTM",
    "dataset": "dataset/processed/03_preprocessed.csv (lstm_text column)",
    "accuracy": metrics["accuracy"],
    "precision": metrics["precision"],
    "recall": metrics["recall"],
    "f1_score": metrics["f1_score"],
    "training_time_seconds": training_time_seconds,
    "vocabulary_size": VOCAB_SIZE,
    "notes": (
        f"AUC={auc_score:.4f}; epochs_completed={epochs_completed}/{LSTM_MAX_EPOCHS}; "
        f"best_val_loss={best_val_loss:.4f}; best_val_accuracy={best_val_accuracy:.4f}; "
        f"embedding_dim={EMBEDDING_DIM}; lstm_units={LSTM_UNITS}; max_sequence_length={MAX_SEQUENCE_LENGTH}"
    ),
}

experiments_df = log_experiment(EXPERIMENTS_LOG_FILE, experiment_record)
experiments_df


,timestamp,model,dataset,accuracy,precision,recall,f1_score,training_time_seconds,vocabulary_size,notes
0,2026-07-19T17:25:38.084284+00:00,TF-IDF + Logistic Regression (baseline),dataset/processed/03_preprocessed.csv (baseline_text column),0.982402,0.978836,0.989305,0.984043,0.245526,20000,AUC=0.9979; unigrams only; stop words + lemmatization applied in Stage 3
1,2026-07-19T17:41:48.661544+00:00,TF-IDF + Logistic Regression (Reuters-ablation experiment),"dataset/processed/03_preprocessed.csv (baseline_text, 'reuters' token removed)",0.982574,0.979738,0.988676,0.984187,0.226553,20000,"AUC=0.9979; ablation study, NOT the official baseline; McNemar exact p=1.0000 vs official baseli..."
2,2026-07-19T18:24:56.757077+00:00,LSTM,dataset/processed/03_preprocessed.csv (lstm_text column),0.972050,0.975717,0.973262,0.974488,599.123938,20000,AUC=0.9917; epochs_completed=10/10; best_val_loss=0.1100; best_val_accuracy=0.9703; embedding_di...


## Model Persistence

Saved under `models/lstm/`: the trained model (`model.keras`), the fitted tokenizer
(`tokenizer.pkl` — required, since `/predict` in Phase 7 must turn new article text into the
exact same integer sequences this model was trained on), a label map, and metadata (mirroring
`models/baseline/`'s structure so both models are loaded the same way later).


In [15]:
save_lstm_artifacts(
    model, tokenizer, LSTM_MODEL_DIR,
    extra_metadata={
        "test_set_metrics": metrics,
        "test_set_auc": auc_score,
        "epochs_completed": epochs_completed,
        "best_val_loss": best_val_loss,
        "best_val_accuracy": best_val_accuracy,
        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "test_rows": len(test_df),
    },
)
print("Saved:", sorted(p.name for p in LSTM_MODEL_DIR.iterdir()))


Saved: ['label_map.json', 'metadata.json', 'model.keras', 'tokenizer.pkl']


## Summary

- **Pipeline:** `lstm_text` → Tokenizer (20,000-word vocabulary, fit on train only) → padded
  sequences (length 300) → Embedding(100) → SpatialDropout1D → LSTM(64) → Dropout → Dense(32,
  ReLU) → Dense(1, sigmoid).
- **Training:** up to 10 epochs, EarlyStopping on validation loss (patience 3,
  restore_best_weights), ModelCheckpoint saving the best model.
- **Test-set results:** see the Evaluation section above and `models/lstm/metadata.json` for
  the exact numbers from this run.
- All artifacts needed to reuse this model without retraining are saved under `models/lstm/`.

## What this notebook deliberately did not do

**No formal baseline-vs-LSTM comparison was made** — this phase's objective was to build a
complete, correct, well-documented LSTM classifier, not to declare a winner. The rigorous,
statistically-grounded comparison (following the same McNemar's-test approach already used in
`docs/reuters_ablation_study.md`) is Phase 6's job, once both models' results exist side by
side. The official baseline in `models/baseline/` was not read, modified, or retrained anywhere
in this notebook.
